In [174]:
# Import all necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.preprocessing import MinMaxScaler,MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer

In [175]:
df = pd.read_csv("anime.csv")
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [176]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


In [177]:
df.isnull().sum()

anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

In [178]:
# Convert numeric-like columns (e.g., 'Unknown') to NaN
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df['members'] = pd.to_numeric(df['members'], errors='coerce')

In [179]:
# Handle missing values
df['genre'] = df['genre'].fillna('').str.lower().str.replace(' ', '')
df['rating'] = df['rating'].fillna(df['rating'].mean())
df['members'] = df['members'].fillna(df['members'].median())
df['episodes'] = df['episodes'].replace('Unknown', np.nan)
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')
df['episodes'] = df['episodes'].fillna(df['episodes'].median())
df['type'] = df['type'].fillna(df['type'].mode()[0])

In [180]:
df.isnull().sum()

anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64

In [181]:
# TF-IDF for genres
df['genre'] = df['genre'].fillna('Unknown').str.replace(',', ' ')
tfidf = TfidfVectorizer(token_pattern=r'[^,]+')
tfidf_matrix = tfidf.fit_transform(df['genre'])

In [182]:
#  Normalize numeric features
scaler = MinMaxScaler()
num_features = scaler.fit_transform(df[['rating', 'members', 'episodes']])

In [183]:
#  Combine genre and numeric features (weighted)
combined_features = hstack([tfidf_matrix * 0.6, num_features * 0.4])


In [184]:
#  Compute cosine similarity
cosine_sim = cosine_similarity(combined_features, combined_features)
print("\nFeatures combined and cosine similarity matrix computed!")


Features combined and cosine similarity matrix computed!


In [193]:
cosine_sim = cosine_similarity(combined_features, combined_features)

In [192]:
def recommend_anime(title, df, cosine_sim, top_n=10):
    
    if title not in df['name'].values:
        return f"'{title}' not found in the dataset."

    # Get index of the target anime
    index = df[df['name'] == title].index[0]

    # Get similarity scores for all anime
    scores = list(enumerate(cosine_sim[index]))

    # Sort scores in descending order (skip self)
    sorted_scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:top_n+1]

    # Get indices of top similar anime
    anime_indices = [i[0] for i in sorted_scores]

    return df.iloc[anime_indices][['name', 'genre', 'rating', 'members']]

# Example test
print("\n Example Recommendations for 'Naruto':")
print(recommend_anime("Naruto", df, cosine_sim, top_n=5))


 Example Recommendations for 'Naruto':
                                                   name  \
615                                  Naruto: Shippuuden   
1472        Naruto: Shippuuden Movie 4 - The Lost Tower   
1573  Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...   
486                            Boruto: Naruto the Movie   
1343                                        Naruto x UT   

                                             genre  rating  members  
615   action comedy martialarts shounen superpower    7.94   533578  
1472  action comedy martialarts shounen superpower    7.53    84527  
1573  action comedy martialarts shounen superpower    7.50    83515  
486   action comedy martialarts shounen superpower    8.03    74690  
1343  action comedy martialarts shounen superpower    7.58    23465  


In [194]:
def evaluate_recommendation_system(df, cosine_sim, top_n=10, sample_size=20, overlap_threshold=0.05):
    
    np.random.seed(42)
    sampled_df = df.sample(min(sample_size, len(df)))

    precisions, recalls, f1s = [], [], []

    for _, anime in sampled_df.iterrows():
        title = anime['name']
        true_genres = set(str(anime['genre']).split(','))

        # Get recommendations
        recs = recommend_anime(title, df, cosine_sim, top_n)
        if isinstance(recs, str) or recs.empty:
            continue

        # Define ground truth similar anime (≥ overlap_threshold)
        similarities = []
        for _, other in df.iterrows():
            if other['name'] == title:
                continue
            other_genres = set(str(other['genre']).split(','))
            overlap = len(true_genres & other_genres) / max(len(true_genres), 1)
            if overlap >= overlap_threshold:
                similarities.append(other['name'])

        predicted = recs['name'].tolist()

        # Binary relevance arrays
        y_true = [1 if anime in similarities else 0 for anime in df['name']]
        y_pred = [1 if anime in predicted else 0 for anime in df['name']]

        precisions.append(precision_score(y_true, y_pred, zero_division=0))
        recalls.append(recall_score(y_true, y_pred, zero_division=0))
        f1s.append(f1_score(y_true, y_pred, zero_division=0))

    print("\n Improved Evaluation Results:")
    print(f"Average Precision: {np.mean(precisions):.2f}")
    print(f"Average Recall:    {np.mean(recalls):.2f}")
    print(f"Average F1-score:  {np.mean(f1s):.2f}")
    print(f"Samples Evaluated: {len(precisions)}")

In [189]:
evaluate_recommendation_system(df, cosine_sim, top_n=20, sample_size=30, overlap_threshold=0.05)


 Improved Evaluation Results:
Average Precision: 0.53
Average Recall:    0.47
Average F1-score:  0.34
Samples Evaluated: 30


#### Difference between User-Based and Item-Based Collaborative Filtering

###### User-based collaborative filtering recommends items by finding users with similar tastes and suggesting what they liked. In contrast, item-based collaborative filtering finds items similar to the ones a user has already liked and recommends them. Simply put, user-based focuses on finding similar users, while item-based focuses on finding similar items. Item-based methods are generally more stable and scalable because item relationships change less frequently than user preferences.

#### What is Collaborative Filtering, and How Does It Work:

###### Collaborative filtering is a recommendation technique that predicts a user’s interests based on the preferences of many users. It works by identifying patterns in user behavior—if two users liked similar items in the past, the system assumes they will like similar items in the future. The method uses similarity measures such as cosine similarity or Pearson correlation on a user-item rating matrix to suggest the most relevant items.